# production_health_dashboard_manifest_gate_v1

验证 `src/schedule` 的 production health 是否默认要求 dashboard export 带有强 `manifest_source`。强来源只允许 `analysis_manifest` 或 `artifact_manifest`，并且需要 `path`、`manifest_hash`、`producer` 齐全。

In [1]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

EXP = "production_health_dashboard_manifest_gate_v1"
ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.core import DatasetSnapshot, publish_production_manifest
from src.schedule import ProductionHealthCheckConfig, run_production_health_check_once

WORKDIR = ROOT / "research" / EXP
RUNTIME = ROOT / "runtime" / "research" / EXP
if RUNTIME.exists():
    shutil.rmtree(RUNTIME)
RUNTIME.mkdir(parents=True, exist_ok=True)

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def publish_strategy(base_dir: Path, dashboard_payload: dict) -> tuple[Path, Path]:
    card = base_dir / "production" / "paper_sim_replay_v1"
    write_json(card / "production_summary.json", {"status": "paper_replay_completed"})
    write_json(card / "dashboard_export.json", dashboard_payload)
    write_json(card / "intent_ledger.json", {"schema_version": "api.intent_ledger.v1"})
    write_json(card / "paper_execution_result.json", {"status": "paper_executed"})
    write_json(card / "data" / "data_registry.json", {"schema_version": "data.registry.v1"})
    write_json(card / "matrix_card" / "analysis_manifest.json", {"schema_version": "analysis.manifest.v1"})
    (card / "paper_sim_replay_v1.ipynb").write_text("{}", encoding="utf-8")
    dataset = DatasetSnapshot(
        name="paper_replay_soxl_daily",
        universe=("SOXL",),
        start="2026-01-02",
        end="2026-06-09",
        calendar_path="data/qlib/soxl/calendars/day.txt",
        data_paths=("data/data_registry.json",),
    )
    publish_production_manifest(
        strategy_id="paper_sim_replay_v1",
        output_dir=card,
        registry_path=base_dir / "production" / "production_registry.json",
        notebook_path="paper_sim_replay_v1.ipynb",
        dataset=dataset,
        paths={
            "summary": "production_summary.json",
            "dashboard_export": "dashboard_export.json",
            "intent_ledger": "intent_ledger.json",
            "paper_execution_result": "paper_execution_result.json",
            "data_registry": "data/data_registry.json",
            "analysis_manifest": "matrix_card/analysis_manifest.json",
        },
        metadata={
            "status": "paper_replay_completed",
            "mode": "paper",
            "live_broker_enabled": False,
            "paper_execution_pipeline_schema_version": "schedule.paper_execution.v1",
            "paper_execution_job_status": "success",
        },
        published_at="2026-06-09T21:30:00Z",
    )
    return card, base_dir / "production" / "production_registry.json"


In [2]:
strong_manifest_source = {
    "kind": "analysis_manifest",
    "path": "matrix_card/analysis_manifest.json",
    "manifest_hash": "a" * 64,
    "producer": "src.analysis.exports",
}
cases = {
    "strong_analysis_manifest": {
        "dashboard_payload": {
            "schema_version": "dashboard.model_matrix.v1",
            "manifest_source": strong_manifest_source,
        },
        "expected_health_status": "healthy",
        "expected_dashboard_manifest_status": "complete",
    },
    "missing_manifest_source": {
        "dashboard_payload": {"schema_version": "dashboard.model_matrix.v1"},
        "expected_health_status": "dashboard_manifest_source_missing",
        "expected_dashboard_manifest_status": "missing",
    },
    "fixed_sidecar_manifest_source": {
        "dashboard_payload": {
            "schema_version": "dashboard.model_matrix.v1",
            "manifest_source": {
                "kind": "fixed_sidecars",
                "path": "",
                "manifest_hash": "",
                "producer": "src.dashboard.model_matrix",
            },
        },
        "expected_health_status": "dashboard_manifest_source_weak",
        "expected_dashboard_manifest_status": "weak_kind",
    },
    "missing_manifest_hash": {
        "dashboard_payload": {
            "schema_version": "dashboard.model_matrix.v1",
            "manifest_source": {
                "kind": "analysis_manifest",
                "path": "matrix_card/analysis_manifest.json",
                "manifest_hash": "",
                "producer": "src.analysis.exports",
            },
        },
        "expected_health_status": "dashboard_manifest_source_incomplete",
        "expected_dashboard_manifest_status": "incomplete",
    },
}

results = []
for case_name, spec in cases.items():
    case_dir = RUNTIME / case_name
    _, registry_path = publish_strategy(case_dir, spec["dashboard_payload"])
    events: list[str] = []
    config = ProductionHealthCheckConfig(
        name="production_health_check",
        registry_path=registry_path,
        strategy_id="paper_sim_replay_v1",
        output_path=case_dir / "production_health.json",
        notify_details=True,
    )
    result = run_production_health_check_once(config, lock_dir=case_dir / "locks", notify=events.append)
    payload = result.payload
    assert result.status == "success"
    assert payload["health_status"] == spec["expected_health_status"]
    assert payload["dashboard_manifest_status"] == spec["expected_dashboard_manifest_status"]
    results.append(
        {
            "case": case_name,
            "health_status": payload["health_status"],
            "is_healthy": payload["is_healthy"],
            "dashboard_manifest_required": payload["dashboard_manifest_required"],
            "dashboard_manifest_status": payload["dashboard_manifest_status"],
            "dashboard_manifest_kind": payload["dashboard_manifest_kind"],
            "dashboard_manifest_hash_present": bool(payload["dashboard_manifest_hash"]),
            "dashboard_manifest_producer": payload["dashboard_manifest_producer"],
            "notification": events[0] if events else "",
        }
    )

summary = {
    "schema_version": "research.production_health_dashboard_manifest_gate_v1",
    "experiment": EXP,
    "healthy_case_count": sum(1 for item in results if item["is_healthy"]),
    "blocked_case_count": sum(1 for item in results if not item["is_healthy"]),
    "cases": results,
}
write_json(WORKDIR / "analysis_summary.json", summary)
summary


{'schema_version': 'research.production_health_dashboard_manifest_gate_v1',
 'experiment': 'production_health_dashboard_manifest_gate_v1',
 'healthy_case_count': 1,
 'blocked_case_count': 3,
 'cases': [{'case': 'strong_analysis_manifest',
   'health_status': 'healthy',
   'is_healthy': True,
   'dashboard_manifest_required': True,
   'dashboard_manifest_status': 'complete',
   'dashboard_manifest_kind': 'analysis_manifest',
   'dashboard_manifest_hash_present': True,
   'dashboard_manifest_producer': 'src.analysis.exports',
   'notification': 'production_health_check:success | strategy=paper_sim_replay_v1 | health=healthy | missing=0 | dashboard_manifest=analysis_manifest/complete | paper_execution=schedule.paper_execution.v1/success | manifest=/Users/rziiiii/dev/workspace/runtime/research/production_health_dashboard_manifest_gate_v1/strong_analysis_manifest/production/paper_sim_replay_v1/production_manifest.json'},
  {'case': 'missing_manifest_source',
   'health_status': 'dashboard_m

## 中文结论

- 做了什么：验证 production health 默认检查 dashboard export 的 `manifest_source`，并把缺失来源、`fixed_sidecars` 来源、缺少 `manifest_hash` 的来源判定为不健康。
- 核心指标：4 个合成场景中，只有 `analysis_manifest` 且 `path`、`manifest_hash`、`producer` 齐全的场景为 healthy；其余 3 个场景分别被标记为 `dashboard_manifest_source_missing`、`dashboard_manifest_source_weak`、`dashboard_manifest_source_incomplete`。
- regime 差异：本次是调度/manifest 契约验证，不直接比较市场 regime；意义在于后续 regime 分层和模型对比只能通过 manifest 声明的公开分析结果进入 dashboard 和生产检查。
- 是否晋升：该能力应进入 `src/schedule` 作为 paper/production 健康门禁；没有新增 production strategy。
- 风险：当前只检查 dashboard export 自身声明的 `manifest_source` 字段完整性，不递归校验 analysis manifest 中每个 artifact 的业务内容。
- 下一步：继续把 production/paper API、schedule dashboard refresh 和 dashboard page 都收敛到同一份 manifest/export 证据链。